In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from keras.models import load_model
from keras.layers import concatenate, Dense
from tensorflow.keras import regularizers, Model, Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import Callback, EarlyStopping

## Simulation4b
English   Id + Pos--> orth

But here the data is non-case-sensitive (upper and lower case are the same)

In [ ]:
X_id  = pd.read_excel('Non_case_Sensitive_word_encodings.xlsx', sheet_name='identity',
                      index_col=0, engine='openpyxl').values.astype('float32')
X_pos = pd.read_excel('Non_case_Sensitive_word_encodings.xlsx', sheet_name='position',
                      index_col=0, engine='openpyxl').values.astype('float32')

Y     = pd.read_excel('Non_case_Sensitive_orthograph_encodings.xlsx', sheet_name='orth_vector',
                       engine='openpyxl').values.astype('float32')


in_id   = Input(shape=(26,),  name='in_id')
tower_id  = Dense(700, activation='sigmoid')(in_id)

in_pos  = Input(shape=(100,), name='in_pos')
tower_pos = Dense(700, activation='sigmoid')(in_pos)

combined = concatenate([tower_id, tower_pos])
h        = Dense(500, activation='sigmoid')(combined)
output   = Dense(260, activation='sigmoid', name='orth_out')(h)

model4b = Model(inputs=[in_id, in_pos], outputs=output)


model4b.compile(optimizer='adam', loss='mse')


es = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)

model4b.fit([X_id, X_pos], Y,
          batch_size=25,
          epochs=700,
          callbacks=[es])

model4b.save('simulation4b_english_model.h5')


Epoch 1/700
19/19 ━━━━━━━━━━━━━━━━━━━━ 5s 47ms/step - loss: 0.1264
Epoch 2/700
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 41ms/step - loss: 0.0262
Epoch 3/700
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - loss: 0.0260
Epoch 4/700
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - loss: 0.0259
Epoch 5/700
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - loss: 0.0259
Epoch 6/700
19/19 ━━━━━━━━━━━━━━━━━━━━ 2s 55ms/step - loss: 0.0258
Epoch 7/700
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - loss: 0.0255
Epoch 8/700
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0258
Epoch 9/700
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0258
Epoch 10/700
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0257
Epoch 11/700
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0257
Epoch 12/700
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.0255
Epoch 13/700
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0257
Epoch 14/700
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 0.0257
Epoch 15/700
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - lo

# Test the model
We will find the correlation distance between the original word and its different modified forms.

In [ ]:
Y     = pd.read_excel('Non_case_Sensitive_orthograph_encodings.xlsx', sheet_name='orth_vector',
                       engine='openpyxl').values.astype('float32')


data_lst= ["Non_case_Sensitive_TL_Same_Case", "Non_case_Sensitive_TL_DF_Case", "Non_case_Sensitive_SL_Same_Case", "Non_case_Sensitive_SL_DF_Case"]
df_sim5 = pd.DataFrame()
for document in data_lst:
    X_id   = pd.read_excel(document+".xlsx", sheet_name='identity', index_col=0).values.astype(np.float32)
    X_pos  = pd.read_excel(document+".xlsx", sheet_name='position', index_col=0).values.astype(np.float32)
    pred = model4b.predict([X_id, X_pos])
    valz = []
    for i in range(len(Y)):
      r = np.corrcoef(pred[i], Y[i])[0,1]
      valz.append(round(1-r,4))


    df_sim5[f'target-{document}']= valz

df_sim5.to_excel('simulation4b_correlations_test_10.xlsx',index=False)

15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step


# Correlation Distance Between Words that share the same stem

In [ ]:
import itertools

df_words    = pd.read_excel('words_with_vectors.xlsx', sheet_name='words',     index_col=0)
words    = df_words.iloc[:,0].values
pred_all = model4b.predict([X_id, X_pos])

df_pred = pd.DataFrame(pred_all, columns=[f'v{i}' for i in range(pred_all.shape[1])])
df_pred['word'] = words


# Load roots file
df_roots = pd.read_excel('words_with_vectors.xlsx', sheet_name='grouped_by_stem',     index_col=0)


# Merge so each row has its vector
df_merged = df_roots.merge(df_pred, on='word', how='inner')


# compute all pairwise distances for each root
rows = []
for root, grp in df_merged.groupby('stem'):
    items = list(zip(grp['word'], grp[[f'v{i}' for i in range(pred_all.shape[1])]].values))
    for (w1, vec1), (w2, vec2) in itertools.combinations(items, 2):
        r = np.corrcoef(vec1, vec2)[0,1]
        dist = 1 - r
        rows.append({'stem': root, 'word1': w1, 'word2': w2, 'distance': dist})

df_pairs = pd.DataFrame(rows)

# Optionally - summary stats per root
summary = df_pairs.groupby('stem')['distance'].agg(
    mean_dist='mean',
    std_dist ='std',
    min_dist ='min',
    max_dist ='max'
).reset_index()


with pd.ExcelWriter('simulation4b_stem_pairwise_distances_10.xlsx') as wr:
    df_pairs.to_excel(wr, sheet_name='all_pairs', index=False)
    summary.to_excel(wr, sheet_name='per_stem_summary', index=False)



15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step


# Correlation Distance Between the word and other 4 conditions

In [ ]:
import numpy as np
import pandas as pd

df_words    = pd.read_excel('words_with_vectors.xlsx', sheet_name='words',     index_col=0)
words    = df_words.iloc[:,0].values
pred_all = model4b.predict([X_id, X_pos])



df_pred = pd.DataFrame(pred_all, columns=[f'v{i}' for i in range(pred_all.shape[1])])
df_pred['word'] = words




PAIRS_XLSX  = 'words_with_vectors.xlsx'
PAIRS_SHEET = '4_conditions'


vec_cols = [c for c in df_pred.columns if c.startswith('v')]
vec_map  = dict(zip(df_pred['word'], df_pred[vec_cols].values))

# read pairs
pairs = pd.read_excel(PAIRS_XLSX, sheet_name=PAIRS_SHEET)

def corr_distance(w1, w2):
    v1 = vec_map.get(w1)
    v2 = vec_map.get(w2)
    if v1 is None or v2 is None:
        return np.nan
    r = np.corrcoef(v1, v2)[0, 1]
    return 1 - r

pairs['distance'] = pairs.apply(lambda r: corr_distance(r['word1'], r['word2']), axis=1)


out = pairs[['word1', 'word2', 'condition', 'distance']]
out.to_excel('simulation4b_4_conditions_pairs_with_distance_10.xlsx', index=False)



15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
